# Lab 3 — Data Quality Plan (DQP)

**核心目标**
1. 读取原始训练集 `ppr-group-25208508-train.csv`
2. 按规则清洗（类型、缺失、重复、异常值、业务规则）
3. 产出：
   - `ppr-group-25208508-clean-final.csv`（清洗后的最终数据）
   - 关键步骤日志



## 1) Imports & Settings
统一导入依赖，并设置 pandas 显示选项。

In [38]:
import re
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 200)

## 2) Load Dataset
读取训练集 `ppr-group-25208508-train.csv`。

In [39]:
from pathlib import Path

DATA_PRIMARY = Path("ppr-group-25208508-train.csv")

if DATA_PRIMARY.exists():
    data_path = DATA_PRIMARY
else:
    raise FileNotFoundError("Cannot find ppr-group-25208508-train.csv (or fallback preview).")

df_raw = pd.read_csv(data_path)
print("Loaded:", data_path)
print("Shape:", df_raw.shape)
df_raw.head()

Loaded: ppr-group-25208508-train.csv
Shape: (54000, 9)


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,30/09/2016,"28 BRACKEN COURT, DONNYBROOK, CORK",Cork,NaN,"€181,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,20/12/2016,"2 AN CLOCHAR, CONVENT RD, DONERAILE",Cork,NaN,"€50,152.49",No,Yes,New Dwelling house /Apartment,less than 38 sq metres
2,28/09/2016,"Apartment 7 The Court, Clonattin, Gorey",Wexford,NaN,"€62,171.81",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
3,16/09/2016,"6 Monalin, Wicklow Hills, Newtownmountkennedy",Wicklow,NaN,"€223,348.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
4,29/01/2016,"18 Lislea, Frascati Park, Blackrock",Dublin,NaN,"€310,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN


## 3) Quick Structure Check (Before Cleaning)
输出列名、缺失、类型、重复行数量，作为“清洗前基线”。

In [40]:
def quick_profile(df: pd.DataFrame, title: str):
    print("="*80)
    print(title)
    print("Shape:", df.shape)
    print("\nDtypes:")
    display(df.dtypes.to_frame("dtype"))
    print("\nMissing (top 15):")
    miss = df.isna().sum().sort_values(ascending=False)
    display(miss.head(15).to_frame("missing_count"))
    print("\nDuplicate rows:", int(df.duplicated().sum()))
    print("="*80)

quick_profile(df_raw, "BEFORE CLEANING")

BEFORE CLEANING
Shape: (54000, 9)

Dtypes:


,dtype
Date of Sale (dd/mm/yyyy),str
Address,str
County,str
Eircode,str
Price (€),str
Not Full Market Price,str
VAT Exclusive,str
Description of Property,str
Property Size Description,str



Missing (top 15):


,missing_count
Property Size Description,51164
Eircode,37108
Date of Sale (dd/mm/yyyy),0
County,0
Address,0
Price (€),0
Not Full Market Price,0
VAT Exclusive,0
Description of Property,0



Duplicate rows: 9


## 4) Utility Functions
用于标准化文本、价格解析、Address 特征提取、日志记录等。

In [41]:
from dataclasses import dataclass, field

@dataclass
class DQLog:
    steps: list = field(default_factory=list)

    def add(self, step: str, before_shape, after_shape, notes: str = ""):
        br, bc = before_shape
        ar, ac = after_shape
        self.steps.append({
            "step": step,
            "rows_before": br, "rows_after": ar, "rows_delta": ar - br,
            "cols_before": bc, "cols_after": ac, "cols_delta": ac - bc,
            "notes": notes
        })

    def to_frame(self):
        return pd.DataFrame(self.steps)

log = DQLog()

def clean_whitespace(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"\s+", " ", s)
    return s

def parse_price_to_numeric(series: pd.Series) -> pd.Series:
    # Handles values like "€123,456.78" or "123,456"
    s = series.astype(str).str.strip()
    s = s.str.replace("€", "", regex=False).str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

def extract_dublin_district(addr: str):
    """Extract Dublin district if pattern appears.
    Examples: 'Dublin 4', 'Dublin 6W', 'D6', 'D6W' -> 'Dublin 4' / 'Dublin 6W'
    Returns NaN if not found.
    """
    if pd.isna(addr):
        return np.nan
    s = str(addr).lower()
    m = re.search(r"\bdublin\s*(\d{1,2}w?)\b", s)
    if m:
        return "Dublin " + m.group(1).upper()
    m = re.search(r"\bd(\d{1,2}w?)\b", s)  # e.g., D6W / D4
    if m:
        return "Dublin " + m.group(1).upper()
    return np.nan

def extract_area_token(addr: str):
    """Heuristic area token from Address without external APIs.
    - Split by commas
    - Take the 2nd last token if available (often locality/suburb), else last
    - Normalize to lowercase alphanum + space + hyphen
    """
    if pd.isna(addr):
        return np.nan
    parts = [p.strip() for p in str(addr).split(",") if p.strip()]
    if len(parts) == 0:
        return np.nan
    cand = parts[-2] if len(parts) >= 2 else parts[-1]
    cand = cand.lower()
    cand = re.sub(r"[^a-z0-9\s\-]", "", cand)
    cand = re.sub(r"\s+", " ", cand).strip()
    return cand if cand else np.nan

## 5) Start Cleaning — Work on a Copy
不直接改 df_raw，先复制为 df。

In [42]:
df = df_raw.copy()

## 6) Basic Standardization
- 统一文本字段空格
- 日期字段解析
- 价格字段转为数值
- 创建时间特征（与 DQR 保持一致：Sale Year / Sale Month）

In [43]:
before = df.shape

# Text columns: trim and normalize spaces
text_cols = [c for c in df.columns if df[c].dtype == "object"]
for c in text_cols:
    df[c] = df[c].apply(clean_whitespace)

# Date parsing
date_col = "Date of Sale (dd/mm/yyyy)"
if date_col in df.columns:
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors="coerce")

# Price parsing
price_col = "Price (€)"
if price_col in df.columns:
    df[price_col] = parse_price_to_numeric(df[price_col])

# Simple time features (match DQR naming)
if date_col in df.columns:
    df["Sale Year"] = df[date_col].dt.year
    df["Sale Month"] = df[date_col].dt.month

after = df.shape
log.add("Standardize text + parse date + parse price + create Sale Year/Month", before, after)
df.head()

,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description,Sale Year,Sale Month
0,2016-09-30,"28 BRACKEN COURT, DONNYBROOK, CORK",Cork,NaN,181000.00,No,No,Second-Hand Dwelling house /Apartment,NaN,2016,9
1,2016-12-20,"2 AN CLOCHAR, CONVENT RD, DONERAILE",Cork,NaN,50152.49,No,Yes,New Dwelling house /Apartment,less than 38 sq metres,2016,12
2,2016-09-28,"Apartment 7 The Court, Clonattin, Gorey",Wexford,NaN,62171.81,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,2016,9
3,2016-09-16,"6 Monalin, Wicklow Hills, Newtownmountkennedy",Wicklow,NaN,223348.00,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,2016,9
4,2016-01-29,"18 Lislea, Frascati Park, Blackrock",Dublin,NaN,310000.00,No,No,Second-Hand Dwelling house /Apartment,NaN,2016,1


## 7) Drop Fully Empty / Constant Columns (If Any)
如果存在全空列或常数列，会降低分析价值，直接删除并记录。

In [44]:
before = df.shape

# Fully empty columns
empty_cols = [c for c in df.columns if df[c].isna().all()]
# Constant columns (single unique non-null value)
const_cols = []
for c in df.columns:
    nun = df[c].nunique(dropna=True)
    if nun == 1 and not df[c].isna().all():
        const_cols.append(c)

drop_cols = sorted(set(empty_cols + const_cols))
if drop_cols:
    df = df.drop(columns=drop_cols)

after = df.shape
notes = f"empty_cols={empty_cols}; const_cols={const_cols}"
log.add("Drop fully-empty / constant columns", before, after, notes)
drop_cols

[]

## 8) Duplicate Rows
去掉完全重复的记录，并记录数量。

In [45]:
before = df.shape
dup_count = int(df.duplicated().sum())
df = df.drop_duplicates()
after = df.shape
log.add("Drop duplicate rows", before, after, f"duplicates_removed={dup_count}")
dup_count

9

## 9) Core Field Validations
最基础的数据有效性过滤：
- `Price (€)` 必须为正数
- 日期必须可解析
- County 不能为空

In [46]:
before = df.shape

reject_mask = pd.Series(False, index=df.index)

# price must be > 0
if price_col in df.columns:
    reject_mask |= df[price_col].isna() | (df[price_col] <= 0)

# date must be valid
if date_col in df.columns:
    reject_mask |= df[date_col].isna()

# county not null (optional but usually reasonable)
if "County" in df.columns:
    reject_mask |= df["County"].isna() | (df["County"].astype(str).str.strip() == "")

reject_count = int(reject_mask.sum())
df_rejected = df.loc[reject_mask].copy()
df = df.loc[~reject_mask].copy()

after = df.shape
log.add("Filter invalid core rows (price/date/county)", before, after, f"rejected_rows={reject_count}")

print("Rejected rows:", reject_count)
df_rejected.head()

Rejected rows: 0


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description,Sale Year,Sale Month


## 10) Address Handling — Feature Engineering then Drop Raw Address
痛点：
- `County` 过粗
- `Eircode` 缺失较多
- 无法使用付费/免费限制的地址转 Eircode API

**方案 **：
1) 保留 Address 仅用于生成少量、可解释的“粗粒度位置/形态”特征
2) 然后删除原始 Address 列（避免高基数/噪声/隐私风险）

生成的特征：
- `is_apartment`：地址中是否出现 apartment/apt/unit/flat
- `has_number`：是否包含门牌数字
- `dublin_district`：抓取 Dublin 邮区（如 Dublin 4 / D6W）
- `area_token`：不依赖 API 的近似 locality token（逗号切分的倒数第二段）
- `*_missing`：对提取失败的情况用缺失标记


In [47]:
before = df.shape

addr_col = "Address"
if addr_col in df.columns:
    # Basic normalization already done; extract features
    df["is_apartment"] = df[addr_col].str.contains(r"\b(apartment|apt|unit|flat)\b", case=False, na=False).astype(int)
    df["has_number"] = df[addr_col].str.contains(r"\b\d+\b", na=False).astype(int)

    df["dublin_district"] = df[addr_col].apply(extract_dublin_district)
    df["dublin_district_missing"] = df["dublin_district"].isna().astype(int)

    df["area_token"] = df[addr_col].apply(extract_area_token)
    df["area_token_missing"] = df["area_token"].isna().astype(int)

    # Finally drop raw Address
    df = df.drop(columns=[addr_col])

after = df.shape
log.add("Scheme A: derive address features then drop Address", before, after, "Added: is_apartment, has_number, dublin_district, area_token + missing flags")
df.head()

C:\Users\Derek\AppData\Local\Temp\ipykernel_16416\3198737329.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["is_apartment"] = df[addr_col].str.contains(r"\b(apartment|apt|unit|flat)\b", case=False, na=False).astype(int)


,Date of Sale (dd/mm/yyyy),County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description,Sale Year,Sale Month,is_apartment,has_number,dublin_district,dublin_district_missing,area_token,area_token_missing
0,2016-09-30,Cork,NaN,181000.00,No,No,Second-Hand Dwelling house /Apartment,NaN,2016,9,0,1,NaN,1,donnybrook,0
1,2016-12-20,Cork,NaN,50152.49,No,Yes,New Dwelling house /Apartment,less than 38 sq metres,2016,12,0,1,NaN,1,convent rd,0
2,2016-09-28,Wexford,NaN,62171.81,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,2016,9,1,1,NaN,1,clonattin,0
3,2016-09-16,Wicklow,NaN,223348.00,No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...,2016,9,0,1,NaN,1,wicklow hills,0
4,2016-01-29,Dublin,NaN,310000.00,No,No,Second-Hand Dwelling house /Apartment,NaN,2016,1,0,1,NaN,1,frascati park,0


## 11) Price Outliers — Clamping (As in DQR)
用 **5%–95%** 分位数做 price clamping（截断极端值）。

这里实现为：
- 计算 `Price (€)` 的 p05 与 p95
- 生成新列 `Price_Clamped`（保留原价列，便于审计）
- 同时记录被截断的条数

In [48]:
before = df.shape

if price_col in df.columns:
    p05 = df[price_col].quantile(0.05)
    p95 = df[price_col].quantile(0.95)

    df["Price_Clamped"] = df[price_col].clip(lower=p05, upper=p95)

    clamped_mask = (df["Price_Clamped"] != df[price_col])
    clamped_count = int(clamped_mask.sum())
    notes = f"p05={p05:.2f}, p95={p95:.2f}, clamped_rows={clamped_count}"

after = df.shape
log.add("Price clamping: create Price_Clamped using p05–p95", before, after, notes if price_col in df.columns else "Price column missing")
notes if price_col in df.columns else None

'p05=60000.00, p95=703000.00, clamped_rows=5356'

## 12) VAT Rule (From DQR To-Do)
DQR 提到：对 `Description of Property == 'New Dwelling house /Apartment'` 且 `VAT Exclusive == Yes` 的记录，价格可以按 VAT（13.5%）调整。

这里采用“可审计”的做法：
- 保留 `Price (€)` 原价
- 新增 `Price_Adjusted_VAT`：仅对满足条件的行做 `Price * 1.135`
- 新增 `vat_adjusted_flag`：标记是否被调整

In [49]:
before = df.shape

VAT_RATE = 0.135
df["vat_adjusted_flag"] = 0

if price_col in df.columns and "Description of Property" in df.columns and "VAT Exclusive" in df.columns:
    cond = (
        (df["Description of Property"].astype(str).str.strip() == "New Dwelling house /Apartment") &
        (df["VAT Exclusive"].astype(str).str.strip().str.lower() == "yes") &
        (df[price_col].notna())
    )
    df["Price_Adjusted_VAT"] = df[price_col].copy()
    df.loc[cond, "Price_Adjusted_VAT"] = df.loc[cond, price_col] * (1 + VAT_RATE)
    df.loc[cond, "vat_adjusted_flag"] = 1
    affected = int(cond.sum())
    note = f"VAT_RATE={VAT_RATE}, affected_rows={affected}"
else:
    df["Price_Adjusted_VAT"] = df.get(price_col, np.nan)
    note = "Required columns missing; created Price_Adjusted_VAT as copy/NaN."

after = df.shape
log.add("VAT adjustment: create Price_Adjusted_VAT + vat_adjusted_flag", before, after, note)
note

'VAT_RATE=0.135, affected_rows=9513'

## 13) High Missingness Columns — Keep vs Drop
DQR 显示某些列（例如 `Property Size Description`、`Eircode`）可能缺失较高。

DQP 中采用“更可解释、更稳定”的策略：
- **如果缺失极高且对分析价值很低**：考虑 drop
- **如果缺失高但仍可能提供信息**：保留 + `missing flag` + 合理填充（例如 'Unknown'）

> 设置一个可调阈值 `DROP_MISSING_PCT`。默认 90%。

In [50]:
before = df.shape

DROP_MISSING_PCT = 65.0

missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
candidates = missing_pct[missing_pct >= DROP_MISSING_PCT].index.tolist()

# For transparency, we DO NOT auto-drop everything blindly.
# We drop only columns that are BOTH high-missing and clearly low-utility free-text.
auto_drop_allowlist = set([
    # Example: if present and extremely sparse, usually low utility
    "Property Size Description",
    "dublin_district",
    "Eircode"
])

to_drop = [c for c in candidates if c in auto_drop_allowlist]

if to_drop:
    df = df.drop(columns=to_drop)

after = df.shape
log.add("High-missing columns: optional drop (threshold-based)", before, after, f"DROP_MISSING_PCT={DROP_MISSING_PCT}, candidates={len(candidates)}, dropped={to_drop}")
missing_pct.head(15).to_frame("missing_%")

,missing_%
Property Size Description,94.749125
dublin_district,83.432424
Eircode,68.715156
County,0.000000
Date of Sale (dd/mm/yyyy),0.000000
VAT Exclusive,0.000000
Price (€),0.000000
Description of Property,0.000000
Sale Year,0.000000
Sale Month,0.000000


## 14) Fill Missing Values (Conservative)
为了让下游分析/模型更稳定：
- 类别字段：填充 `'Unknown'`
- 数值字段：填充中位数



In [51]:
before = df.shape

# Identify numeric vs categorical/object
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in df.columns if c not in num_cols]

# Fill categorical
for c in cat_cols:
    # Skip date column if still datetime (should not be in cat)
    if c == date_col:
        continue
    df[c] = df[c].fillna("Unknown")

# Fill numeric
for c in num_cols:
    if df[c].isna().any():
        med = df[c].median()
        df[c] = df[c].fillna(med)

after = df.shape
log.add("Fill missing values (cat='Unknown', num=median)", before, after)
df.head()

,Date of Sale (dd/mm/yyyy),County,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Sale Year,Sale Month,is_apartment,has_number,dublin_district_missing,area_token,area_token_missing,Price_Clamped,vat_adjusted_flag,Price_Adjusted_VAT
0,2016-09-30,Cork,181000.00,No,No,Second-Hand Dwelling house /Apartment,2016,9,0,1,1,donnybrook,0,181000.00,0,181000.00000
1,2016-12-20,Cork,50152.49,No,Yes,New Dwelling house /Apartment,2016,12,0,1,1,convent rd,0,60000.00,1,56923.07615
2,2016-09-28,Wexford,62171.81,No,Yes,New Dwelling house /Apartment,2016,9,1,1,1,clonattin,0,62171.81,1,70565.00435
3,2016-09-16,Wicklow,223348.00,No,Yes,New Dwelling house /Apartment,2016,9,0,1,1,wicklow hills,0,223348.00,1,253499.98000
4,2016-01-29,Dublin,310000.00,No,No,Second-Hand Dwelling house /Apartment,2016,1,0,1,1,frascati park,0,310000.00,0,310000.00000


## 15) Post-Clean Validation
检查：
- 是否仍有缺失
- 价格列是否全部为正
- 关键衍生列是否存在


In [52]:
quick_profile(df, "AFTER CLEANING (POST-VALIDATION)")

# Core checks
assert (df[price_col] > 0).all(), "Found non-positive prices after cleaning."
if date_col in df.columns:
    assert df[date_col].notna().all(), "Found invalid dates after cleaning."

# Show remaining missing
remaining_missing = df.isna().sum().sum()
print("Total remaining missing values:", int(remaining_missing))

AFTER CLEANING (POST-VALIDATION)
Shape: (53991, 16)

Dtypes:


,dtype
Date of Sale (dd/mm/yyyy),datetime64[us]
County,str
Price (€),float64
Not Full Market Price,str
VAT Exclusive,str
Description of Property,str
Sale Year,int32
Sale Month,int32
is_apartment,int64
has_number,int64



Missing (top 15):


,missing_count
Date of Sale (dd/mm/yyyy),0
County,0
Price (€),0
Not Full Market Price,0
VAT Exclusive,0
Description of Property,0
Sale Year,0
Sale Month,0
is_apartment,0
has_number,0



Duplicate rows: 668
Total remaining missing values: 0


## 16) DQ Processing Log
每一步清洗对行/列的影响。

In [53]:
log_df = log.to_frame()
display(log_df)

print("Final shape:", df.shape)

,step,rows_before,rows_after,rows_delta,cols_before,cols_after,cols_delta,notes
0,Standardize text + parse date + parse price + ...,54000,54000,0,9,11,2,
1,Drop fully-empty / constant columns,54000,54000,0,11,11,0,empty_cols=[]; const_cols=[]
2,Drop duplicate rows,54000,53991,-9,11,11,0,duplicates_removed=9
3,Filter invalid core rows (price/date/county),53991,53991,0,11,11,0,rejected_rows=0
4,Scheme A: derive address features then drop Ad...,53991,53991,0,11,16,5,"Added: is_apartment, has_number, dublin_distri..."
5,Price clamping: create Price_Clamped using p05...,53991,53991,0,16,17,1,"p05=60000.00, p95=703000.00, clamped_rows=5356"
6,VAT adjustment: create Price_Adjusted_VAT + va...,53991,53991,0,17,19,2,"VAT_RATE=0.135, affected_rows=9513"
7,High-missing columns: optional drop (threshold...,53991,53991,0,19,16,-3,"DROP_MISSING_PCT=65.0, candidates=3, dropped=[..."
8,"Fill missing values (cat='Unknown', num=median)",53991,53991,0,16,16,0,


Final shape: (53991, 16)


## 17) Export Clean Dataset
导出最终清洗结果到 `ppr-group-25208508-clean-final.csv`。

In [54]:
OUT_CSV = "ppr-group-25208508-clean-final.csv"
df.to_csv(OUT_CSV, index=False)
print("Saved:", OUT_CSV)

Saved: ppr-group-25208508-clean-final.csv


## 18) Quick Peek of Output
抽样检查导出后的数据是否符合预期。

In [55]:
df.sample(5, random_state=42)

,Date of Sale (dd/mm/yyyy),County,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Sale Year,Sale Month,is_apartment,has_number,dublin_district_missing,area_token,area_token_missing,Price_Clamped,vat_adjusted_flag,Price_Adjusted_VAT
45584,2023-09-15,Westmeath,299950.0,No,No,Second-Hand Dwelling house /Apartment,2023,9,0,1,1,lakepoint park,0,299950.0,0,299950.0
21937,2019-04-26,Cavan,130000.0,No,No,Second-Hand Dwelling house /Apartment,2019,4,0,1,1,swellan,0,130000.0,0,130000.0
1319,2016-12-20,Dublin,209000.0,No,No,Second-Hand Dwelling house /Apartment,2016,12,0,0,1,balgriffin,0,209000.0,0,209000.0
4640,2016-07-11,Galway,93000.0,No,No,Second-Hand Dwelling house /Apartment,2016,7,0,0,1,doughuiska,0,93000.0,0,93000.0
46394,2023-06-14,Wexford,102500.0,No,No,Second-Hand Dwelling house /Apartment,2023,6,0,1,1,faythe,0,102500.0,0,102500.0
